# 中研院 strength estimator 推論

重現 `aicup/strength_se.py` 的推論，輸出每局目標玩家各手的 SE∞ `score`（每種對稱各一組平均與標準差）與 SL 11 級機率的平均。

- runtime：A100 GPU
- 外部資源：rlglab/strength-estimator 的 `go.tar.gz`（GPL-3.0），由程式從官方網址下載，只用 `se_infty_go.pt`、`sl_go.pt`
- Drive 的 `MyDrive/aicup-2026/` 要有 `training-v1.1/` 的 10 個 CSV（官方訓練集 v1.1，含 `HD[n]`）與 `humansl-analysis/val648-p3.jsonl`
- `test` 模式另外要有 `tests/rank_prediction_test_{public,private}.csv`，以及 Drive 上已完成的 `humansl-analysis/val648-se.jsonl`（取它的 dtype）
- Colab Secrets 要有唯讀的 `GH_TOKEN`

Drive 的做法：

- 不跑 `aicup.colab_setup`。原因：它會讀 `rank-features/`，這份推論用不到特徵。
- CSV 先複製到 `/content`，推論期間不讀 Drive。
- 結果寫在 `/content`，每批結束後複製到 Drive。斷線時最多損失最近一批，重新開 runtime 依序執行所有 cell 就會接續。
- 推論中不要中斷 cell。

## 參數

`MODE` 選 `val`、`test`。

- `val`：和 `val648-p3.jsonl` 相同的 12,960 局，8 種對稱
- `test`：競賽二的測試題，對稱數依 val 的 stacking 結果設定 `SYM`，dtype 沿用 val 的輸出
- `BF16_CHECK`：只用於 `val`。先用前 64 局比對 bf16 與 fp32，通過才採用 bf16

In [ ]:
MODE = 'val'
SYM = 8
BATCH = 1000
BF16_CHECK = True

DRIVE = '/content/drive/MyDrive/aicup-2026'
RUNS = {  # (輸入參數, 輸出檔名, 總局數)
    'val': ('--keys-from /content/val648-p3.jsonl', 'val648-se.jsonl', 12960),
    'test': ('--test-dir /content/tests', 'test-se.jsonl', 10000),
}
SRC, OUT_NAME, TOTAL = RUNS[MODE]
COMMON = "--csv '/content/training/train_*.csv' --model-dir /content/se-models"

## 取得程式

In [ ]:
import os, subprocess
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('pull', '-q')
!pip install -q sgfmill
!git log --oneline -1
!nvidia-smi -L
os.environ['DRIVE'], os.environ['OUT_NAME'], os.environ['MODE'] = DRIVE, OUT_NAME, MODE

## 複製資料

要看到 `10` 與 `val648-p3.jsonl`，`test` 模式還要看到 2 個測試 CSV 與 `val648-se.jsonl`。Drive 上已有結果時會複製回本地，接續推論。

In [ ]:
%%bash
mkdir -p /content/training
cp "$DRIVE"/training-v1.1/train_*.csv /content/training/
cp "$DRIVE/humansl-analysis/val648-p3.jsonl" /content/
if [ "$MODE" = test ]; then
    mkdir -p /content/tests
    cp "$DRIVE"/tests/rank_prediction_test_*.csv /content/tests/
    cp "$DRIVE/humansl-analysis/val648-se.jsonl" /content/val648-se-ref.jsonl
    ls -la /content/tests /content/val648-se-ref.jsonl
fi
if [ -f "$DRIVE/humansl-analysis/$OUT_NAME" ]; then
    cp "$DRIVE/humansl-analysis/$OUT_NAME" /content/
fi
ls /content/training | wc -l
ls -la /content/val648-p3.jsonl

## 冒煙測試與 dtype

`val` 模式用前 64 局跑 fp32，第一次執行會下載 `go.tar.gz`（約 450 MB）。`BF16_CHECK` 時再跑 bf16 比對：每局 `score_mean` 的最大差小於 0.05、`sl_rank` 的最大差小於 0.005 才採用 bf16。最後印出速度、估計時間與推論要用的參數。

`test` 模式不比對，dtype 沿用 `val648-se.jsonl`，val 與 test 的設定才會一致。

In [ ]:
import json, re
import numpy as np


def run(args, out):
    lines = !python -m aicup.strength_se {args} --out {out} 2>&1
    lines = [l for l in lines if 'warn' not in l.lower()]
    print('\n'.join(lines[-3:]))
    speed = [float(m.group(1)) for l in lines for m in [re.search(r'每秒 (\d+) 個局面', l)] if m]
    if not speed:
        raise SystemExit('沒有跑完，看上面的錯誤訊息')
    return speed[-1]


def rows(path):
    return {r['key']: r for r in map(json.loads, open(path))}


if MODE == 'val':
    SMOKE = "--keys-from /content/val648-p3.jsonl %s --symmetries %d --limit 64" % (COMMON, SYM)
    !rm -f /content/smoke-fp32.jsonl /content/smoke-bf16.jsonl
    speed = run(SMOKE, '/content/smoke-fp32.jsonl')
    DTYPE = 'float32'
    a = rows('/content/smoke-fp32.jsonl')
    if BF16_CHECK:
        speed16 = run(SMOKE + ' --dtype bfloat16', '/content/smoke-bf16.jsonl')
        b = rows('/content/smoke-bf16.jsonl')
        ds = max(np.abs(np.subtract(a[k]['score_mean'], b[k]['score_mean'])).max() for k in a)
        dr = max(np.abs(np.subtract(a[k]['sl_rank'], b[k]['sl_rank'])).max() for k in a)
        print('bf16 對 fp32：%d 局，score_mean 最大差 %.4f，sl_rank 最大差 %.4f，速度 %.0f 對 %.0f'
              % (len(a), ds, dr, speed16, speed))
        if len(b) == len(a) and ds < 0.05 and dr < 0.005:
            DTYPE, speed = 'bfloat16', speed16
            print('採用 bf16')
        else:
            print('不採用 bf16，維持 float32')
    per_game = np.mean([r['n'] for r in a.values()])
    print('估計 %.1f 小時（每局 %.0f 個局面，每秒 %.0f 個局面）' % (TOTAL * per_game / speed / 3600, per_game, speed))
else:
    DTYPE = json.loads(open('/content/val648-se-ref.jsonl').readline())['dtype']
    print('沿用 val 的 dtype：%s' % DTYPE)
ARGS = '%s %s --dtype %s --symmetries %d' % (SRC, COMMON, DTYPE, SYM)
print(ARGS)

## 推論

每批結束會印出 Drive 上的行數，數字增加代表備份成功。全部跑完時行數等於總局數。

In [ ]:
LOCAL = '/content/' + OUT_NAME
LOG = '/content/%s-se.log' % MODE
BACKUP = DRIVE + '/humansl-analysis'
for n in sorted({*range(BATCH, TOTAL, BATCH), TOTAL}):
    !python -m aicup.strength_se {ARGS} --limit {n} --out {LOCAL} 2>&1 | grep -v -i warn | tee -a {LOG}
    !mkdir -p {BACKUP} && cp {LOCAL} {LOG} {BACKUP}/ && wc -l {BACKUP}/{OUT_NAME}

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣額度。

In [ ]:
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()